## Gold Layer: Star Schema Export for Power BI Analytics

The star schema bridges analytical outputs from five upstream notebooks into a unified Power BI data model. Each table is exported at the grain required for its analytical purpose - transactional detail for order-level measures, dimension tables for filtering and hierarchy, and pre-aggregated fact tables for decision-grade visualizations that cannot be computed from raw order lines alone.

### Core Star Schema

- **FactOrders:** Order-line grain. Carries transactional measures (sales, profit, discount, quantity), engineered lead-time metrics, late-delivery risk scores, and foreign keys to all dimension tables. This is the central fact table; all order-level measures in Power BI originate here.

- **DimDate:** Calendar dimension spanning the full order date range. Provides year, quarter, month, and week attributes for time-intelligence hierarchies.

- **DimCustomer:** One row per customer. Contains demographic attributes, location data, and the CustomerKey for joining to FactOrders and DimCustomerSegments.

- **DimProduct:** Product master with category and department hierarchies for assortment and margin analysis.

- **DimGeo:** Geographic hierarchy at the market × region × country grain. Powers regional filtering and corridor drill-down. Composite GeoKey ensures stable single-column relationships.

### Upstream Analytical Outputs

- **DimCustomerSegments:** RFM-based behavioral clusters imported from the Customer-Centric Operations notebook. Enables cohort analysis by customer loyalty and value tier without recomputing segmentation logic in Power BI.

- **FactWeeklyForecast:** Demand projections at the market-week grain, imported from the Demand Forecasting notebook. Supports forward-looking capacity planning against historical fulfillment performance.

- **DimShippingMode:** Shipping mode dimension augmented with SLA calibration fields from the Global Logistics analysis. Carries observed lead-time distributions (P50, P90), late rates, implied SLA thresholds, and computed SLA gap metrics. This table directly powers the Promise vs. Physics visualization - the central finding that late delivery is driven by SLA miscalibration, not logistics capability.

- **FactBottleneck:** Segment-level bottleneck scores at the Market × Region × Mode grain. Each row is a ranked intervention priority combining tail exposure, observed lateness probability, and volume-weighted material impact. This table powers the intervention shortlist visualization. Segments are classified into Critical, Elevated, Moderate, and Low priority tiers based on composite score thresholds.

- **LeadTimeMRM:** Full Market × Region × Mode segmentation detail for drill-down analysis. Provides the disaggregated lead-time and late-rate data behind the bottleneck ranking.

### Integration Logic

The schema supports three analytical workflows:

- **SLA Calibration:** `DimShippingMode` carries the mode-level performance data and implied SLA thresholds needed to quantify the gap between delivery physics and service promises.

- **Intervention Prioritization:** `FactBottleneck `provides a sequenced, scored list of Mode × Corridor segments - sorted by composite bottleneck score - so operational resources target the highest-impact segments first.

- **Drill-Down Validation:** LeadTimeMRM preserves the three-way segmentation detail so any bottleneck segment can be inspected for its underlying lead-time distribution, late rate, and risk score alignment.

In [ ]:
# =============================================================================
# POWER BI STAR SCHEMA EXPORT
# SCOR Alignment: Enable (Cross-Process Performance Management)
#
# Builds a dimensional model (star schema) from the silver orders dataset and
# integrates analytical outputs from all upstream notebooks:
#   - Demand Forecasting (Plan)
#   - Operational Risk Modeling (Source → Make → Deliver)
#   - Customer Segmentation (Deliver → Return)
#   - Global Logistics & Bottleneck Analysis (Deliver)
#
# Output tables are written as Parquet files for direct ingestion into Power BI.
# =============================================================================

import pandas as pd
import numpy as np
import os

# =============================================================================
# 1. LOAD SILVER ORDERS DATASET
# =============================================================================
# The silver table is the canonical order-line-level source. All downstream
# aggregations, segmentations, and risk scores from prior notebooks are joined
# into this dataset before dimensional decomposition.

orders_silver_risk = pd.read_parquet("/kaggle/input/datasets/orders-silver-with-risk/orders_silver_with_risk.parquet")
df = orders_silver_risk.copy()

# =============================================================================
# 2. STANDARDIZE CORE NUMERIC FIELDS
# =============================================================================
# Coerce financial and quantity columns to numeric types with safe defaults.
# Handles schema variations across dataset versions without breaking downstream
# aggregations.

df["sales_num"]  = pd.to_numeric(df.get("sales_per_customer", df.get("sales", 0)), errors="coerce").fillna(0)
df["profit_num"] = pd.to_numeric(df.get("order_profit_per_order", 0), errors = "coerce").fillna(0)
df["disc_num"]   = pd.to_numeric(df.get("order_item_discount", 0), errors = "coerce").fillna(0)
df["qty_num"]    = pd.to_numeric(df.get("order_item_quantity", 1), errors = "coerce").fillna(1)

# =============================================================================
# 3. LEAD TIME ENGINEERING
# =============================================================================
# Primary metric: calculated transit duration from shipment to delivery
# confirmation. Falls back to dataset-provided column if timestamp fields
# are unavailablee.
#
# Capped variant (99.5th percentile) is generated for visualization stability
# only. The uncapped delivery_days column remains the analytical source of truth
# for all statistical calculations.

if "delivery_days" not in df.columns:
    if "shipping_date" in df.columns and "actual_delivery_date" in df.columns:
        df["delivery_days"] = (df["actual_delivery_date"] - df["shipping_date"]).dt.days
    else:
        df["delivery_days"] = np.nan

# # Generate capped auxiliary column for plotting
if "delivery_days_capped" not in df.columns:
    cap = df["delivery_days"].dropna().quantile(0.995) if df["delivery_days"].notna().any() else 0
    df["delivery_days_capped"] = df["delivery_days"].clip(lower = 0, upper = cap)

# =============================================================================
# 4. DIMENSIONAL KEY GENERATION
# =============================================================================
# Surrogate and composite keys are generated at the row level to enable clean
# star schema relationships.

# ----- DateKey: integer representation (YYYYMMDD) for DimDate join -----
if "order_date" in df.columns:
    df["DateKey"] = df["order_date"].dt.strftime("%Y%m%d").astype(int)
else:
    df["DateKey"] = np.nan

# ----- GeoKey: composite market|region|country for DimGeo join -----
# Uses order_country preferentially; falls back to customer_country.
geo_country = "order_country" if "order_country" in df.columns else ("customer_country" if "customer_country" in df.columns else None)

df["GeoKey"] = (
    df["market"].astype(str) + "|" +
    df["order_region"].astype(str) + "|" +
    (df[geo_country].astype(str) if geo_country else "NA")
)

# ----- ProductKey: product_card_id for DimProduct join -----
product_key_col = "product_card_id" if "product_card_id" in df.columns else ("order_item_product_price" if "order_item_product_price" in df.columns else None)
if product_key_col:
    df["ProductKey"] = df[product_key_col].astype(str)
else:
    df["ProductKey"] = "NA"

# ----- CustomerKey: customer identifier for DimCustomer join -----
df["CustomerKey"] = df["customer_id"].astype(int) if "customer_id" in df.columns else df["order_customer_id"].astype(int)

# ----- ShippingModeKey: shipping mode for DimShippingMode join -----
df["ShippingModeKey"] = df["shipping_mode"].astype(str) if "shipping_mode" in df.columns else "NA"

# ----- CorridorKey: origin → destination lane identifier -----
# Captures the directional flow from customer country to fulfillment country.
# Used for logistics corridor analysis.
if geo_country and "customer_country" in df.columns:
    df["CorridorKey"] = df["customer_country"].astype(str) + " → " + df[geo_country].astype(str)
else:
    df["CorridorKey"] = "NA"

# =============================================================================
# 5. BUILD DIMENSION TABLES
# =============================================================================
# Each dimension is built as a distinct table with a primary key column and
# descriptive attributes. Duplicates are removed to enforce dimension grain.

# ----- DimDate: calendar dimension spanning the full order date range -----
# Grain: one row per calendar day. Includes year, month, week, and quarter
# attributes for time-intelligence hierarchies in Power BI.
if "order_date" in df.columns:
    dates = pd.date_range(df["order_date"].min().normalize(), df["order_date"].max().normalize(), freq = "D")
    dim_date = pd.DataFrame({"date": dates})
    dim_date["DateKey"] = dim_date["date"].dt.strftime("%Y%m%d").astype(int)
    dim_date["year"] = dim_date["date"].dt.year
    dim_date["month"] = dim_date["date"].dt.month
    dim_date["month_name"] = dim_date["date"].dt.strftime("%b")
    dim_date["week"] = dim_date["date"].dt.isocalendar().week.astype(int)
    dim_date["quarter"] = dim_date["date"].dt.quarter
else:
    dim_date = pd.DataFrame()

# ----- DimCustomer: customer master with segmentation attributes -----
# Grain: one row per unique customer. Includes geography and segment fields
# for customer-level filtering and cohort analysis.
cust_cols = [c for c in [
    "customer_id","customer_segment","customer_city","customer_state","customer_zipcode","customer_country"
] if c in df.columns]
dim_customer = df[cust_cols].drop_duplicates().copy()
if "customer_id" in dim_customer.columns:
    dim_customer = dim_customer.rename(columns = {"customer_id":"CustomerKey"})

# ----- DimProduct: product master with category hierarchy -----
# Grain: one row per unique product. Includes category and department
# attributes for product-level filtering and margin analysis.
prod_cols = [c for c in [
    "product_card_id","product_name","category_name","department_name","product_price","product_status"
] if c in df.columns]
dim_product = df[prod_cols].drop_duplicates().copy()
if "product_card_id" in dim_product.columns:
    dim_product["ProductKey"] = dim_product["product_card_id"].astype(str)
else:
    dim_product["ProductKey"] = df["ProductKey"]

# ----- DimGeo: geographic hierarchy dimension -----
# Grain: one row per unique market × region × country combination.
# Provides the geographic rollup path for corridor analysis.
geo_cols = [c for c in ["market","order_region"] if c in df.columns]
if geo_country:
    geo_cols.append(geo_country)
dim_geo = df[geo_cols + ["GeoKey"]].drop_duplicates().copy()
dim_geo = dim_geo.rename(columns = {geo_country: "country"}) if geo_country else dim_geo

# =============================================================================
# 6. BUILD FACT ORDERS TABLE
# =============================================================================
# Grain: one row per order line item. Contains transactional measures
# (sales, profit, discount, quantity), lead time metrics, risk scores,
# and foreign keys to all dimension tables.
# This is the central fact table in the star schema.

fact_cols = [c for c in [
    "order_id","order_item_id","DateKey","CustomerKey","ProductKey","GeoKey","ShippingModeKey","CorridorKey",
    "order_date","shipping_date","actual_delivery_date",
    "sales_num","profit_num","disc_num","qty_num",
    "late_delivery_risk","late_risk_score",
    "delivery_days","delivery_days_capped",
    "order_status","type"
] if c in df.columns]

fact = df[fact_cols].copy()

# =============================================================================
# 7. IMPORT UPSTREAM NOTEBOOK OUTPUTS
# =============================================================================
# Analytical outputs from prior notebooks are loaded and integrated into the
# star schema. Each import includes existence checks so the export does not
# fail if an upstream output is unavailable.

# ----- 7a. Customer Segments (Notebook: Customer-Centric Operations) -----
# SCOR Alignment: Deliver → Return (Customer Segmentation)
# RFM-based customer segments for cohort and retention analysis.

file_path_customers_seg = "/kaggle/input/datasets/customer-centric-ops-sc/customer_segments_rfm.parquet"

if os.path.exists(file_path_customers_seg):
    customer_segments = pd.read_parquet(file_path_customers_seg)
    if "customer_id" in customer_segments.columns:
        customer_segments = customer_segments.rename(columns = {"customer_id": "CustomerKey"})
else:
    print(f"Error: File not found at {file_path_customers_seg}")

# ----- 7b. Demand Forecast (Notebook: Demand Forecasting) -----
# SCOR Alignment: Plan (Demand Planning)
# Weekly forecasted demand by market for capacity and inventory planning.

file_path_demand_forecast = "/kaggle/input/datasets/demand-forecasting-markets-sc/FactForecastWeekly.parquet"

if os.path.exists(file_path_demand_forecast):
    forecast_weekly = pd.read_parquet(file_path_demand_forecast)
else:
    print(f"Error: File not found at {file_path_demand_forecast}")

# ----- 7c. Global Logistics Analysis (Notebook: Global Logistics & Bottlenecks) -----
# SCOR Alignment: Deliver (Reliability Management, Intervention Planning)
#
# Three analytical outputs from the corrected SLA-focused bottleneck analysis:
#   - bottleneck_rank.parquet:   Prioritized Mode × Corridor intervention shortlist
#   - mode_performance.parquet:  Mode-level KPIs for SLA calibration
#   - lead_mrm.parquet:          Full Market × Region × Mode segmentation detail
#
# These tables power the Promise vs. Physics SLA Gap and Bottleneck Score visualizations.

bottleneck_path = "/kaggle/input/datasets/global-logistics-sc/bottleneck_rank.parquet"
mode_perf_path  = "/kaggle/input/datasets/global-logistics-sc/mode_performance.parquet"
lead_mrm_path   = "/kaggle/input/datasets/global-logistics-sc/lead_mrm.parquet"

bottleneck_rank = None
mode_performance = None
lead_mrm_detail = None

if os.path.exists(bottleneck_path):
    bottleneck_rank = pd.read_parquet(bottleneck_path)
    print(f"Loaded bottleneck_rank: {len(bottleneck_rank)} segments, "
          f"tiers = {bottleneck_rank['priority_tier'].value_counts().to_dict()}")
else:
    print(f"WARNING: {bottleneck_path} not found. Bottleneck graphs in Power BI will not render.")

if os.path.exists(mode_perf_path):
    mode_performance = pd.read_parquet(mode_perf_path)
    print(f"Loaded mode_performance: {len(mode_performance)} modes")
else:
    print(f"WARNING: {mode_perf_path} not found. SLA Gap graph in Power BI will not render.")

if os.path.exists(lead_mrm_path):
    lead_mrm_detail = pd.read_parquet(lead_mrm_path)
    print(f"Loaded lead_mrm_detail: {len(lead_mrm_detail)} segments (≥200 order floor)")
else:
    print(f"WARNING: {lead_mrm_path} not found. Drill-down support limited.")

# =============================================================================
# 8. BUILD AUGMENTED DIMSHIPPINGMODE (WITH SLA CALIBRATION FIELDS)
# =============================================================================
# SCOR Alignment: Deliver (Service Design, Reliability Engineering)
#
# The shipping mode dimension is augmented with mode-level performance KPIs
# from the Global Logistics analysis. This enables the Promise vs. Physics
# SLA Gap visualization - the central finding that late delivery is driven
# by SLA miscalibration, not logistics capability.
#
# Implied SLA thresholds are inferred from the observed lead time distribution
# and late rate patterns documented in the Global Logistics notebook:
#   - First Class:    p50 = p90 = 2, late_rate = 95.3% → SLA ≈ 1 day
#   - Second Class:   p50 = 4, p90 = 6, late_rate = 76.8% → SLA ≈ 3 days
#   - Standard Class: p50 = 4, p90 = 6, late_rate = 38.1% → SLA ≈ 5 days
#   - Same Day:       p50 = 0, p90 = 1, late_rate = 45.7% → SLA ≈ 0 days

if mode_performance is not None:
    # Build from mode-level performance aggregates
    dim_ship = mode_performance.copy()
    
    # Key column for star schema relationships
    dim_ship["ShippingModeKey"] = dim_ship["shipping_mode"].astype(str)
    
    # Infer SLA thresholds from distribution shape and late rate patterns
    sla_map = {
        "Same Day": 0,
        "First Class": 1,
        "Second Class": 3,
        "Standard Class": 5
    }
    dim_ship["ImpliedSLA_Days"] = dim_ship["shipping_mode"].map(sla_map)
    
    # SLA Gap: positive = network is slower than promised (over-promised)
    dim_ship["SLA_Gap_Days"] = dim_ship["p50_delivery_days"] - dim_ship["ImpliedSLA_Days"]
    
    # SLA Gap Ratio: how many multiples of the promise window delivery actually takes
    # First Class: p50 = 2, SLA = 1 → ratio = 2.0 (delivers at 2× the promised window)
    dim_ship["SLA_Gap_Ratio"] = np.where(
        dim_ship["ImpliedSLA_Days"] > 0,
        dim_ship["p50_delivery_days"] / dim_ship["ImpliedSLA_Days"],
        np.nan  # Same Day has SLA = 0, ratio undefined
    )
    
    print(f"DimShippingMode built with SLA calibration fields: {dim_ship.shape[0]} modes")
else:
    # Fallback: minimal dim from fact table if mode_performance unavailable
    if "shipping_mode" in df.columns:
        dim_ship = df[["shipping_mode"]].drop_duplicates().copy()
        dim_ship["ShippingModeKey"] = dim_ship["shipping_mode"].astype(str)
    else:
        dim_ship = pd.DataFrame()

# =============================================================================
# 9. BUILD FACTBOTTLENECK TABLE
# =============================================================================
# SCOR Alignment: Deliver (Reliability Management, Intervention Sequencing)
#
# Segment-level bottleneck scores at the Market × Region × Mode grain.
# This table powers the Bottleneck Score by Mode × Corridor visualization.
#
# The bottleneck_score formula: P90 × late_rate × log1p(orders).
# This captures tail severity, lateness probability, and volume-weighted
# material impact in a single composite metric.

if bottleneck_rank is not None:
    fact_bottleneck = bottleneck_rank[[
        "market", "order_region", "shipping_mode",
        "orders", "p90_delivery_days", "late_rate", 
        "bottleneck_score", "priority_tier"
    ]].copy()
    
    # Composite geo key for optional relationship to DimGeo
    fact_bottleneck["GeoKey"] = (
        fact_bottleneck["market"].astype(str) + "|" +
        fact_bottleneck["order_region"].astype(str) + "|NA"
    )
    
    # Shipping mode key for relationship to DimShippingMode
    fact_bottleneck["ShippingModeKey"] = fact_bottleneck["shipping_mode"].astype(str)
    
    print(f"FactBottleneck built: {len(fact_bottleneck)} segments, "
          f"Critical = {len(fact_bottleneck[fact_bottleneck['priority_tier'] == 'Critical'])}")
else:
    fact_bottleneck = pd.DataFrame()
    print("WARNING: FactBottleneck not built - bottleneck_rank not loaded.")

# =============================================================================
# 10. SAVE ALL TABLES TO PARQUET
# =============================================================================
# Output files are written to the working directory for Power BI ingestion.
# Each file is saved with a descriptive name matching its dimensional role.

# ----- Core star schema tables -----

file_dim_date = "DimDate.parquet"
dim_date.to_parquet(file_dim_date, index = False)
print(f"File saved successfully: {file_dim_date}")

file_dim_customer = "DimCustomer.parquet"
dim_customer.to_parquet(file_dim_customer, index = False)
print(f"File saved successfully: {file_dim_customer}")

file_dim_product = "DimProduct.parquet"
dim_product.to_parquet(file_dim_product, index = False)
print(f"File saved successfully: {file_dim_product}")

file_dim_geo = "DimGeo.parquet"
dim_geo.to_parquet(file_dim_geo, index = False)
print(f"File saved successfully: {file_dim_geo}")

file_fact = "FactOrders.parquet"
fact.to_parquet(file_fact, index = False)
print(f"File saved successfully: {file_fact}")

# DimShippingMode (augmented with SLA calibration fields)
file_dim_ship_v2 = "DimShippingMode.parquet"
dim_ship.to_parquet(file_dim_ship_v2, index=False)
print(f"File saved: {file_dim_ship_v2} ({len(dim_ship)} modes)")

# ----- Global Logistics analysis tables -----

if not fact_bottleneck.empty:
    file_fact_bn = "FactBottleneck.parquet"
    fact_bottleneck.to_parquet(file_fact_bn, index=False)
    print(f"File saved: {file_fact_bn} ({len(fact_bottleneck)} segments)")

if lead_mrm_detail is not None:
    file_lead_mrm = "LeadTimeMRM.parquet"
    lead_mrm_detail.to_parquet(file_lead_mrm, index=False)
    print(f"File saved: {file_lead_mrm} ({len(lead_mrm_detail)} segments)")

# ----- Upstream analytical outputs -----

if customer_segments is not None:
    file_customer_segments = "DimCustomerSegments.parquet"
    customer_segments.to_parquet(file_customer_segments, index = False)

if forecast_weekly is not None:
    file_forecast_weekly = "FactWeeklyForecast.parquet"
    forecast_weekly.to_parquet(file_forecast_weekly, index = False)

# =============================================================================
# EXPORT COMPLETE
# =============================================================================

print(f"\nStar schema export complete.")
print("FactOrders rows:", len(fact), "| cols:", fact.shape[1])
print(f"Integrated outputs: CustomerSegments, WeeklyForecast, Bottleneck, LeadTimeMRM")

Loaded bottleneck_rank: 50 segments, tiers = {'Critical': 48, 'Elevated': 2}
Loaded mode_performance: 4 modes
Loaded lead_mrm_detail: 61 segments (≥200 order floor)
DimShippingMode built with SLA calibration fields: 4 modes
FactBottleneck built: 50 segments, Critical = 48
File saved successfully: DimDate.parquet
File saved successfully: DimCustomer.parquet
File saved successfully: DimProduct.parquet
File saved successfully: DimGeo.parquet
File saved successfully: FactOrders.parquet
File saved: DimShippingMode.parquet (4 modes)
File saved: FactBottleneck.parquet (50 segments)
File saved: LeadTimeMRM.parquet (61 segments)

Star schema export complete.
FactOrders rows: 159763 | cols: 21
Integrated outputs: CustomerSegments, WeeklyForecast, Bottleneck, LeadTimeMRM
